# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [4]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard
from functions import snspd_counts_vs_current
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Experiment loaded. Last ID no: 472


In [16]:
import functions
import importlib
importlib.reload(functions)
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard

# Instruments

In [7]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Light Counts vs Current

In [17]:
# Take same current range as ID 12
ID = 12
data = load_by_id(ID).get_parameter_data()
print(data['yoko_current']['yoko_current'])

[6.000e-06 6.250e-06 6.500e-06 6.750e-06 7.000e-06 7.250e-06 7.500e-06
 7.750e-06 8.000e-06 8.250e-06 8.500e-06 8.750e-06 9.000e-06 9.250e-06
 9.500e-06 9.750e-06 1.000e-05 1.025e-05 1.050e-05 1.075e-05 1.100e-05
 1.125e-05 1.150e-05 1.175e-05 1.200e-05 1.225e-05 1.250e-05 1.275e-05
 1.300e-05 1.325e-05 1.350e-05 1.375e-05 1.400e-05 1.425e-05]


In [18]:
np.arange(6e-6, 14.5e-6, 0.25e-6)

array([6.000e-06, 6.250e-06, 6.500e-06, 6.750e-06, 7.000e-06, 7.250e-06,
       7.500e-06, 7.750e-06, 8.000e-06, 8.250e-06, 8.500e-06, 8.750e-06,
       9.000e-06, 9.250e-06, 9.500e-06, 9.750e-06, 1.000e-05, 1.025e-05,
       1.050e-05, 1.075e-05, 1.100e-05, 1.125e-05, 1.150e-05, 1.175e-05,
       1.200e-05, 1.225e-05, 1.250e-05, 1.275e-05, 1.300e-05, 1.325e-05,
       1.350e-05, 1.375e-05, 1.400e-05, 1.425e-05])

In [20]:
print(params.v_scale_counts)
print(params.h_time_counts)
print(params.h_pos_counts)

currents = np.arange(6e-6, 14.5e-6, 0.25e-6)

# Set oscilloscope vertical and horizontal parameters 
osc_set_standard(MS, v_scale=params.v_scale_counts, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set attenuator voltage
v_attenuator = 5.3 # from SNSPD4_3_Calibration.ipynb 
p_att.write(f'VOLT {v_attenuator}')

# Set standard laser parameters 
laser_set_standard(laser, 1550e-9, 7)
laser_get_standard(laser)

# Set to start current 
yoko.current(currents[0])

############################ TURN LASER ON ############################ 
laser.enable(True)
time.sleep(10)
print(f'Laser enable status: {laser.enable()}')


snspd_counts_vs_current(MS, dmm, yoko, p_att, device_name=params.device_1_name, n_captures=10, interval=1, currents=currents, station=station)



############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

0.15
0.1
0
MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.15
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0
Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
Laser enable status: True
update station
Ramping current from 6e-06A to 6e-06A
Starting experimental run with id: 475. 
475
Starting current 6e-06
This acquisition will take 10s
14 43
Starting current 6.25e-06
This acquisition will take 10s
14 44
Starting current 6.5e-06
This acquisition will take 10s
14 44
Starting current 6.75e-06
This acquisition will take 10s
14 44
Starting current 7e-06
This acquisition will take 10s
14 44
Starting current 7.25e-06
This acquisition will take 10s
14 45
Starting current 7.5e-06
This acquisition will take 10s
14 45
Starting current 7.75e-06
This acquisition will take 10s
14 45
Starting current 8e-06
This acquisition will take 10s
14 45
Starting current 8.25e-06
This acquisition will take 10s
14 46
Starting current 8.5e-06
This acquisition will take 10s
14 46
Start